In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))
from quantlab.data.binance import load_parquet
from quantlab.features.technical_features import add_returns, add_momentum, add_sma
from quantlab.backtest.costs import apply_transaction_costs
from quantlab.portfolio.allocation import portfolio_returns
from quantlab.backtest.report import performance_summary

%load_ext autoreload
%autoreload 2

In [ ]:
df = load_parquet('../data/raw/BTCUSDT_1d.parquet')
df = add_returns(df)
df = add_momentum(df, 20)
df = add_sma(df, 20)

In [ ]:
# Signals
momentum_20d_signal = pd.Series(np.where(df['momentum_20'] > 0, 1, -1), index=df.index)
momentum_moving_average_signal = pd.Series(np.where(df['close'] > df['sma_20'], 1, -1), index=df.index)

# Strat returns
momentum_20d_return = momentum_20d_signal.shift(1) * df['returns']
momentum_20d_return_net = apply_transaction_costs(momentum_20d_return, momentum_20d_signal, cost_per_trade=0.001)
momentum_moving_average_return = momentum_moving_average_signal.shift(1) * df['returns']
momentum_moving_average_return_net = apply_transaction_costs(momentum_moving_average_return, momentum_moving_average_signal, cost_per_trade=0.001)

In [ ]:
strategy_returns = pd.concat([momentum_20d_return_net, momentum_moving_average_return_net], axis = 1)
strategy_returns.columns = ['20d', 'moving_average']

weights = pd.Series(data=[0.5, 0.5], index = strategy_returns.columns)

portfolio_returns = portfolio_returns(strategy_returns, weights)

In [ ]:
summaries = pd.DataFrame({
    "portfolio": performance_summary(pd.DataFrame({"returns": portfolio_returns})),
    "momentum_20": performance_summary(pd.DataFrame({"returns": momentum_20d_return_net})),
    "ma_momentum": performance_summary(pd.DataFrame({"returns": momentum_moving_average_return_net})),
    "buy_and_hold": performance_summary(pd.DataFrame({"returns": df["returns"]})),
}).T

summaries